# ChatMessageHistory 的使用

记忆存储

In [7]:

# 实例化

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import SystemMessage


history = ChatMessageHistory()
history.add_user_message("你好")
history.add_ai_message("你好！有什么我可以帮助你的吗？")
history.add_message(SystemMessage(content="这是一个系统消息。"))

history.messages

[HumanMessage(content='你好', additional_kwargs={}, response_metadata={}),
 AIMessage(content='你好！有什么我可以帮助你的吗？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 SystemMessage(content='这是一个系统消息。', additional_kwargs={}, response_metadata={})]

对接大模型

In [2]:
# 获取大模型
# chain的使用
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)  # 加载环境变量

prefix = ""
model_name = os.getenv(prefix + "MODEL")
model_base_url = os.getenv(prefix + "BASE_URL")
model_api_key = os.getenv(prefix + "API_KEY")

print(f"Model Name: {model_name}")


chat_llm = ChatOpenAI(model=model_name or "gpt-3.5-turbo", base_url=model_base_url, api_key=model_api_key) # type: ignore

Model Name: free/gpt-5.4-mini


In [3]:
history = ChatMessageHistory()
history.add_user_message("你好")
history.add_ai_message("你好！有什么我可以帮助你的吗？")
history.add_user_message("你能否介绍一下自己吗？")

res = chat_llm.invoke(history.messages)
print(res.content)

当然可以。

我是一个由 OpenAI 训练的人工智能助手，可以帮助你进行很多类型的任务，比如：

- 回答问题、解释概念
- 写作和润色文本
- 翻译中英文及其他语言
- 提供学习、编程、工作建议
- 帮你头脑风暴、整理思路

我没有真实的个人经历或情感，但我会尽力根据你的需求，给出清晰、实用的回答。

如果你愿意，也可以直接告诉我你现在想做什么，我来帮你。


# ConversationBufferMemory 的使用
ConversationBufferMemory 是 LangChain 中的一种内存类型，用于在对话过程中存储和管理对话历史。它可以帮助模型记住之前的对话内容，从而在后续的对话中提供更连贯和相关的回答。

In [4]:
from langchain_classic.memory import ConversationBufferMemory


history = ConversationBufferMemory()

history.save_context({"input": "你好"}, {"output": "你好！有什么我可以帮助你的吗？"})
history.save_context({"input": "帮我回答1+1等于几"}, {"output": "1+1等于2"})

## 获取对话历史
print(history.load_memory_variables({}))

# 返回的字典的key是history
print(history.load_memory_variables({})["history"])

print(history.chat_memory.messages)

{'history': 'Human: 你好\nAI: 你好！有什么我可以帮助你的吗？\nHuman: 帮我回答1+1等于几\nAI: 1+1等于2'}
Human: 你好
AI: 你好！有什么我可以帮助你的吗？
Human: 帮我回答1+1等于几
AI: 1+1等于2
[HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='你好！有什么我可以帮助你的吗？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='帮我回答1+1等于几', additional_kwargs={}, response_metadata={}), AIMessage(content='1+1等于2', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


C:\Users\Spring\AppData\Local\Temp\ipykernel_4476\1014438791.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  history = ConversationBufferMemory()


In [5]:
# 使用 chain 结构（新版推荐：RunnableWithMessageHistory）
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1) 定义提示词模板：包含历史消息 + 当前输入
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有帮助的 AI 助手，请结合历史对话进行回答。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# 2) LCEL chain
chain = prompt | chat_llm

# 3) 会话级历史存储（示例用内存字典；生产可替换 Redis/DB）
store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# 4) 连续两轮对话，验证“记忆”效果
session_id = "demo-user-001"

res1 = chain_with_history.invoke(
    {"input": "你好，我叫小明。"},
    config={"configurable": {"session_id": session_id}},
)
print("第1轮：", res1.content)

res2 = chain_with_history.invoke(
    {"input": "我刚才叫什么名字？"},
    config={"configurable": {"session_id": session_id}},
)
print("第2轮：", res2.content)

第1轮： 你好，小明，很高兴认识你！有什么我可以帮你的吗？
第2轮： 你刚才叫**小明**。


In [6]:
print("历史消息：", store[session_id].messages)

历史消息： [HumanMessage(content='你好，我叫小明。', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，小明，很高兴认识你！有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 34, 'total_tokens': 54, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'resp_037fbb85c2c603450169c3e429d5408191ae9a99fda501c8af', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2533-3bfe-7853-a779-1d749f40e3fe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 34, 'output_tokens': 20, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}}), HumanMessage(content='我刚才叫什么名字？', additional_kwargs={